# Zero-shot melody match demo

Runs the **same code path** as `zero_shot_nsynth_melody_match.ipynb` and Figure 3: NSynth melody-match triplets, CE-SSL CochCNN9 (`ssl λ=0.5`, `relu4`), and `run_triplet_evaluation` from `lightning_scripts/zero_shot_utils.py`.

**Cluster vs notebook:** full layerwise zero-shot sweeps (all tasks/layers) belong on the cluster — submit `sbatch slurm_scripts/eval_cochdnn_layerwise_zero_shot.sh` and load merged CSVs from `results_dfs/`. This notebook runs a **small** in-memory demo (`N_EXAMPLES=8`) when checkpoints and NSynth are available; it does not replace the SLURM evaluation pipeline.

Requires `COCHDNN_CHECKPOINT_DIR` (defaults to `model_checkpoints/`) and `COCHDNN_NSYNTH_DIR` pointing at a local NSynth tree with `nsynth-valid/examples.json`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "demo_notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from default_paths import MODEL_CHECKPOINT_DIR, NSYNTH_DIR, WORKING_DIRECTORY, require_path
from lightning_scripts.nsynth_triplet_dataset import NsynthTripletDataset
from lightning_scripts.utils.model_build_utils import get_model
from lightning_scripts.zero_shot_utils import MODEL_SR, run_triplet_evaluation

LAYER = "relu4"  # same layer as zero_shot_nsynth_melody_match.ipynb / Figure 3
N_EXAMPLES = 8
CE_SSL_CONFIG = (
    WORKING_DIRECTORY
    / "model_configs/kell2018_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_5e-01.yaml"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Checkpoint dir: {MODEL_CHECKPOINT_DIR}")

model = get_model(CE_SSL_CONFIG, layer_out=LAYER)
model.to(DEVICE)
models = {"ssl_λ=0.5": model}
model_name_map = {"ssl_λ=0.5": "CochCNN9 ssl λ=0.5"}
print(f"Loaded {model_name_map['ssl_λ=0.5']} at layer {LAYER}")

In [ ]:
require_path(NSYNTH_DIR, "COCHDNN_NSYNTH_DIR", "NSynth dataset")

melody_ds = NsynthTripletDataset(
    n_examples=N_EXAMPLES,
    seed=42,
    target_sr=MODEL_SR,
    min_midi=30,
    max_midi=90,
    experiment_type="melody_match",
    balance_eval=True,
)
print(f"Triplet dataset ready: {len(melody_ds)} examples at target_sr={melody_ds.target_sr}")


def nsynth_triplet_collate(batch):
    clips, sr, triplet = batch[0]
    return clips, sr, triplet


def extract_triplet_meta(triplet):
    interval = triplet["anchor"]["interval"]
    negative_interval = triplet["negative"]["interval"]
    return {
        "interval": interval,
        "instrument": triplet["anchor"]["instrument"],
        "negative_interval": negative_interval,
        "interval_diff": negative_interval - interval,
    }


loader = torch.utils.data.DataLoader(
    melody_ds, batch_size=1, shuffle=False, collate_fn=nsynth_triplet_collate
)

results = run_triplet_evaluation(
    models,
    model_name_map,
    loader,
    device=DEVICE,
    model_sr=MODEL_SR,
    include_cosine=True,
    extract_triplet_meta_fn=extract_triplet_meta,
    desc="NSynth melody match (demo)",
)
results

In [ ]:
display_cols = [
    "batch_idx", "model_name", "interval", "instrument",
    "pos_l2", "neg_l2", "judgement_pos_lt_neg",
]
print(
    f"Mean hit rate ({model_name_map['ssl_λ=0.5']}, {LAYER}): "
    f"{results['judgement_pos_lt_neg'].mean():.3f}"
)
results[display_cols]